In [1]:
import pandas as pd
import numpy as np

In [2]:
train_v2 = pd.read_csv('Data/train_v2.csv')
members_v3 = pd.read_csv('Data/members_v3.csv')  
transactions_v2 = pd.read_csv('Data/transactions_v2.csv')
user_v2 = pd.read_parquet('Data/user_logs_v2.parquet')

# train_v2


In [3]:
# train_v2.isnull().sum()
train_v2.duplicated().sum()

np.int64(0)

In [4]:
train_v2['msno'].duplicated().sum()

np.int64(0)

In [5]:
train_v2['is_churn'].value_counts(
 normalize=True
)

is_churn
0    0.910058
1    0.089942
Name: proportion, dtype: float64

# members_v3

In [6]:
members_v3.info()

<class 'pandas.DataFrame'>
RangeIndex: 6769473 entries, 0 to 6769472
Data columns (total 6 columns):
 #   Column                  Dtype
---  ------                  -----
 0   msno                    str  
 1   city                    int64
 2   bd                      int64
 3   gender                  str  
 4   registered_via          int64
 5   registration_init_time  int64
dtypes: int64(4), str(2)
memory usage: 605.9 MB


In [7]:
members_v3.duplicated().sum()

np.int64(0)

In [8]:
members_v3['bd'].value_counts()

bd
 0       4540215
 22       112200
 21       110574
 20       110452
 27       102769
          ...   
 353           1
 954           1
-524           1
-3152          1
 573           1
Name: count, Length: 386, dtype: int64

## bd

In [9]:
#chuyển các giá trị bd < 10 và > 100 thành Nan để dễ xử lý hơn
# - không drop những giá trị vô lý vì nó vẫn có thể có ý nghĩa ở những cột khác để dự đoán 
members_v3['bd']=members_v3['bd'].mask(
  (members_v3['bd']<10)|
  (members_v3['bd']>100),
  np.nan
)

## gender

In [10]:
members_v3['gender'] = (members_v3['gender'].fillna('Unknown').astype('category'))

## city


In [11]:
members_v3['city'].value_counts()

city
1     4804326
5      385069
13     320978
4      246848
22     210407
15     190213
6      135200
14      89940
12      66843
9       47639
11      47489
8       45975
18      38039
10      32482
21      30837
17      27772
3       27282
7       11610
16       5092
20       4233
19       1199
Name: count, dtype: int64

In [12]:
#chuyển về cate vì để dạng int ban đầu dễ bị hiểu là thứ bậc, chuyển về cate
members_v3['city'] = (members_v3['city'].astype('category'))


## Registered_via

- Nên gộp các giá trị về cột Other không

+ count <1000 -> Other

+ đó là dùng thống kê tần suất.

+ nên đợi sau split rồi fit trên train.

+ tránh leakage.

In [13]:
members_v3['registered_via'] = (members_v3['registered_via'].replace(-1,'Unknown').astype('category'))

In [14]:
members_v3['registered_via'].value_counts()

registered_via
4          2793213
3          1643208
9          1482863
7           805895
11           25047
13            5455
8             3982
5             3115
17            1494
2             1452
6             1213
19             974
16             888
14             615
1               43
10              10
18               5
Unknown          1
Name: count, dtype: int64

## Registered_init_time

In [15]:
members_v3['registration_init_time'].unique()

array([20110911, 20110914, 20110915, ..., 20040607, 20040608, 20040525],
      shape=(4782,))

In [16]:
members_v3['registration_init_time'] = pd.to_datetime(members_v3['registration_init_time'],format='%Y%m%d',errors='coerce')

In [17]:
members_v3.isnull().sum()

msno                            0
city                            0
bd                        4546765
gender                          0
registered_via                  0
registration_init_time          0
dtype: int64

In [18]:
members_v3['registration_init_time'].value_counts().sort_index()

registration_init_time
2004-03-26     250
2004-03-27    1481
2004-03-28    1109
2004-03-29     726
2004-03-30     411
              ... 
2017-04-25    1581
2017-04-26    1561
2017-04-27    1526
2017-04-28    1717
2017-04-29    2218
Name: count, Length: 4782, dtype: int64

# transaction_v2

In [19]:
transactions_v2.info()

<class 'pandas.DataFrame'>
RangeIndex: 1431009 entries, 0 to 1431008
Data columns (total 9 columns):
 #   Column                  Non-Null Count    Dtype
---  ------                  --------------    -----
 0   msno                    1431009 non-null  str  
 1   payment_method_id       1431009 non-null  int64
 2   payment_plan_days       1431009 non-null  int64
 3   plan_list_price         1431009 non-null  int64
 4   actual_amount_paid      1431009 non-null  int64
 5   is_auto_renew           1431009 non-null  int64
 6   transaction_date        1431009 non-null  int64
 7   membership_expire_date  1431009 non-null  int64
 8   is_cancel               1431009 non-null  int64
dtypes: int64(8), str(1)
memory usage: 158.3 MB


In [20]:
transactions_v2.duplicated().sum()

np.int64(0)

### transaction_date


In [21]:
transactions_v2['transaction_date'].unique()

array([20170131, 20150809, 20170303, 20170329, 20170323, 20151112,
       20170313, 20170318, 20170316, 20170307, 20170311, 20170326,
       20170331, 20170308, 20150731, 20160805, 20170304, 20170327,
       20170301, 20170228, 20170306, 20170111, 20151120, 20170319,
       20170309, 20151125, 20170314, 20161101, 20170305, 20170322,
       20170310, 20170325, 20161108, 20160307, 20170330, 20170312,
       20170315, 20160402, 20151021, 20151221, 20170107, 20170324,
       20170218, 20170320, 20170302, 20160507, 20151006, 20161202,
       20161226, 20170115, 20150321, 20170205, 20170328, 20151216,
       20161026, 20161121, 20170104, 20161002, 20170317, 20161120,
       20161110, 20150708, 20161210, 20161218, 20151025, 20160829,
       20160327, 20160920, 20161103, 20170125, 20170121, 20160506,
       20170203, 20150808, 20160219, 20170114, 20151213, 20160615,
       20160424, 20161228, 20170212, 20160822, 20150426, 20161010,
       20170321, 20161115, 20170128, 20160408, 20160531, 20161

In [22]:
transactions_v2['membership_expire_date'].unique()

array([20170504, 20190412, 20170422, ..., 20220520, 20240312, 20250519],
      shape=(1960,))

- Convert date

In [23]:
transactions_v2['transaction_date']=pd.to_datetime(
    transactions_v2['transaction_date'],
    format='%Y%m%d',
    errors='coerce'
)

transactions_v2['membership_expire_date']=pd.to_datetime(
    transactions_v2['membership_expire_date'],
    format='%Y%m%d',
    errors='coerce'
)

In [24]:
transactions_v2.isnull().sum()

msno                      0
payment_method_id         0
payment_plan_days         0
plan_list_price           0
actual_amount_paid        0
is_auto_renew             0
transaction_date          0
membership_expire_date    0
is_cancel                 0
dtype: int64

- Check logic


In [25]:
check = transactions_v2[transactions_v2['membership_expire_date'] < transactions_v2['transaction_date']]
check['is_cancel'].value_counts()   

is_cancel
1    5104
0       2
Name: count, dtype: int64

- Theo logic thông thường, khi bạn thanh toán (Transaction Date), ngày hết hạn (Expire Date) phải nằm ở tương lai. Vậy tại sao ở đây nó lại nằm ở quá khứ (Expire < Transaction)?

- => Đây chính xác là hành vi Hủy ngang (Force Cancel). Khi người dùng bấm nút is_cancel = 1, hệ thống của KKBox lập tức cắt dịch vụ của họ. Để làm được việc đó, hệ thống tự động kéo ngược ngày hết hạn về quá khứ (thường là lùi về trước ngày bấm hủy 1 ngày) để vô hiệu hóa tài khoản VIP ngay lập tức.

- Hành động: 5104 dòng này là Dữ liệu Vàng, giữ nguyên, không được xóa hay sửa ngày tháng của chúng. Sự chênh lệch âm (Negative duration) này là tín hiệu cực mạnh báo hiệu Churn.

In [26]:
transactions_v2[
 (transactions_v2['membership_expire_date'] <
  transactions_v2['transaction_date']) &
 (transactions_v2['is_cancel']==0)
]

,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
394563,By0XFlo7S4xoQ+p4OA+rsuScEZ6WT67PCqJJ2fJcm9w=,26,1,0,0,0,2017-03-09,2017-03-08,0
1348445,NpFf1YIXaL3oc2oJ6OF4ngCtlPxWbSHP+JY6YliJzF0=,26,1,0,0,0,2017-03-14,2017-03-12,0


- Hai bản ghi không bị hủy với expire_date sớm hơn transaction_date xuất hiện liên quan đến các gói một ngày miễn phí, cho thấy đây là trường hợp ngoại lệ về gia hạn đăng ký chứ không phải lỗi dữ liệu. Các bản ghi này được giữ lại.

## actual_amount_paid

In [27]:
transactions_v2.info()


<class 'pandas.DataFrame'>
RangeIndex: 1431009 entries, 0 to 1431008
Data columns (total 9 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   msno                    1431009 non-null  str           
 1   payment_method_id       1431009 non-null  int64         
 2   payment_plan_days       1431009 non-null  int64         
 3   plan_list_price         1431009 non-null  int64         
 4   actual_amount_paid      1431009 non-null  int64         
 5   is_auto_renew           1431009 non-null  int64         
 6   transaction_date        1431009 non-null  datetime64[us]
 7   membership_expire_date  1431009 non-null  datetime64[us]
 8   is_cancel               1431009 non-null  int64         
dtypes: datetime64[us](2), int64(6), str(1)
memory usage: 158.3 MB


In [28]:
(transactions_v2['actual_amount_paid']<0).sum()

np.int64(0)

In [29]:
(transactions_v2['plan_list_price']<0).sum()

np.int64(0)

In [30]:
a = transactions_v2[
 transactions_v2['payment_plan_days']<=0
]
a['actual_amount_paid'].value_counts()

actual_amount_paid
149     1702
119      302
129      146
100       20
1788      19
596       15
447        6
1200       5
0          3
Name: count, dtype: int64

1. Tại sao Gói cước 0 ngày mà vẫn có giá 149, 119 NTD?
Insight: Trong số khoảng 2,218 dòng có payment_plan_days = 0, có tới 2,174 dòng (chiếm ~98%) là đi kèm với is_cancel = 1.

Bản chất hệ thống: Đây không phải là khách hàng "mua" gói cước 0 ngày. Đây thực chất là "Biên lai hủy dịch vụ" (Cancellation Receipt) do hệ thống tự động sinh ra.

Khi khách hàng bấm nút "Hủy tự động gia hạn" (Cancel), hệ thống tạo ra một dòng giao dịch (Transaction) mới để ghi nhận sự kiện này. Vì là sự kiện hủy, hệ thống không cấp thêm ngày nào (days = 0), nhưng nó vẫn lưu lại mức giá của gói cước mà khách hàng vừa hủy (ví dụ: hủy gói 149 NTD, hủy gói 119 NTD).

In [31]:
# 1. Tìm danh sách các user (msno) có chứa "biên lai hủy"
cancel_receipt_mask = (transactions_v2['payment_plan_days'] == 0) & (transactions_v2['is_cancel'] == 1)
msno_with_cancel_receipts = transactions_v2[cancel_receipt_mask]['msno'].unique()

print(f"Số lượng user có 'biên lai hủy' (days=0, cancel=1): {len(msno_with_cancel_receipts)}")

# 2. Lấy toàn bộ lịch sử giao dịch của nhóm user này
user_txns = transactions_v2[transactions_v2['msno'].isin(msno_with_cancel_receipts)]

# 3. Đếm xem mỗi user trong nhóm này có bao nhiêu dòng giao dịch
txn_counts = user_txns['msno'].value_counts()
print(f"Số user có NHIỀU HƠN 1 giao dịch: {(txn_counts > 1).sum()}")
print(f"Số user CHỈ CÓ 1 giao dịch: {(txn_counts == 1).sum()}")

# 4. Trích xuất thử 1 user bất kỳ làm bằng chứng (Case study)
if (txn_counts > 1).sum() > 0:
    example_msno = txn_counts[txn_counts > 1].index[0]
    print(f"\n--- Bằng chứng lịch sử giao dịch của user: {example_msno} ---")
    
    # Lọc data của user này và sắp xếp theo ngày
    example_df = user_txns[user_txns['msno'] == example_msno].sort_values('transaction_date')
    
    # Chỉ in ra các cột quan trọng để dễ nhìn
    display(example_df[['msno', 'transaction_date', 'membership_expire_date', 'payment_plan_days', 'actual_amount_paid', 'is_cancel']])

Số lượng user có 'biên lai hủy' (days=0, cancel=1): 44
Số user có NHIỀU HƠN 1 giao dịch: 44
Số user CHỈ CÓ 1 giao dịch: 0

--- Bằng chứng lịch sử giao dịch của user: 8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA= ---


,msno,transaction_date,membership_expire_date,payment_plan_days,actual_amount_paid,is_cancel
952696,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-01-05,2018-05-09,30,119,0
1396495,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-01-09,2018-06-09,30,149,0
65713,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-01-18,2018-07-10,30,149,0
1181869,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-01-25,2018-08-10,30,149,0
903164,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-02-05,2018-09-07,30,119,0
58406,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-02-09,2018-10-05,30,149,0
860498,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-02-18,2018-11-02,30,149,0
1346758,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-02-25,2018-11-30,30,149,0
172507,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-03-05,2018-12-31,30,119,0
1081162,8zVHj2WcT3tUNUOPFDtpG+Q5qYG844bRHKJs1DYNeUA=,2015-03-09,2019-01-31,30,149,0


## is cancel, auto renew - ko có gì 

# User_v2

In [32]:
user_v2.info()

<class 'pandas.DataFrame'>
RangeIndex: 396362 entries, 18000000 to 18396361
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   msno        396362 non-null  str    
 1   date        396362 non-null  int64  
 2   num_25      396362 non-null  int64  
 3   num_50      396362 non-null  int64  
 4   num_75      396362 non-null  int64  
 5   num_985     396362 non-null  int64  
 6   num_100     396362 non-null  int64  
 7   num_unq     396362 non-null  int64  
 8   total_secs  396362 non-null  float64
dtypes: float64(1), int64(7), str(1)
memory usage: 43.8 MB


In [33]:
user_v2['date']=pd.to_datetime(user_v2['date'],format='%Y%m%d',errors='coerce')

In [34]:
user_v2.isnull().sum()

msno          0
date          0
num_25        0
num_50        0
num_75        0
num_985       0
num_100       0
num_unq       0
total_secs    0
dtype: int64

In [35]:
cols = [
'num_25','num_50','num_75',
'num_985','num_100','num_unq', 'total_secs'
]

(user_v2[cols] < 0).sum()

num_25        0
num_50        0
num_75        0
num_985       0
num_100       0
num_unq       0
total_secs    0
dtype: int64

In [36]:
(user_v2['total_secs']==0).sum()

np.int64(0)

- 6. num_unq có thể > tổng plays?

In [37]:
plays = (
 user_v2['num_25'] +
 user_v2['num_50'] +
 user_v2['num_75'] +
 user_v2['num_985'] +
 user_v2['num_100']
)

(user_v2['num_unq'] > plays).sum()

np.int64(0)

# Merge


In [38]:
(
 (transactions_v2['payment_plan_days']==0)
 &
 (transactions_v2['is_cancel']==1)
).sum()

np.int64(44)

- Trong hệ thống KKBox, khi khách hàng bấm hủy gói, hệ thống sinh ra một "Biên lai hủy" (days=0, cancel=1) NHƯNG lại sao chép y nguyên giá trị tiền của gói cước vừa hủy. Nếu không ép số tiền của biên lai này về 0, khi dùng hàm sum() ở bước sau, tổng tiền khách hàng chi trả sẽ bị nhân đôi một cách sai lệch. Dòng lệnh này đảm bảo tính toàn vẹn của dữ liệu tài chính.

In [39]:
# Cutoff
ref_date = pd.to_datetime('2017-03-31')

transactions_v2 = transactions_v2[transactions_v2['transaction_date'] <= ref_date].copy()

user_v2 = user_v2[user_v2['date'] <= ref_date].copy()

In [40]:
tx = transactions_v2.copy()

tx.loc[(tx['payment_plan_days']==0) & (tx['is_cancel']==1),'actual_amount_paid'] = 0

In [41]:
# giao dịch miễn phí
tx['zero_paid'] = (tx['actual_amount_paid']==0).astype(int)

# gói 30 ngày
tx['is_30d'] = (tx['payment_plan_days']==30
).astype(int)

# tỷ lệ giảm giá
tx['discount_rate'] = np.where(
    tx['plan_list_price']>0,
    1 - (tx['actual_amount_paid'] / tx['plan_list_price']),
    np.nan
)

- tạo cờ ở zero_paid và is_30d trước khi gom nhóm giúp chúng ta có thể dễ dàng tính được "Tỷ lệ" (Rate) ở bước sau chỉ bằng một hàm .mean(). Ví dụ: Trung bình của các số (0, 1, 1, 0) sẽ cho ra tỷ lệ 50%.
- Sử dụng np.where thay vì phép chia thông thường để chống lại lỗi ZeroDivisionError. Nếu giá niêm yết (plan_list_price) bằng 0, giá trị sẽ được trả về NaN thay vì làm sập chương trình.

In [42]:
tx = tx[tx['transaction_date'] <= ref_date].copy()
tx['membership_expire_date'] = np.minimum(
    tx['membership_expire_date'],
    ref_date
)

txn_agg = (
    tx.groupby('msno').agg(
      n_txns=('msno','size'),
      cancel_rate=('is_cancel','mean'),
      auto_renew_rate=('is_auto_renew','mean'),
      avg_plan_days=('payment_plan_days','mean'),
      share_30d=('is_30d','mean'),
      zero_paid_rate=('zero_paid','mean'),
      avg_amount_paid=('actual_amount_paid','mean'),
      total_amount_paid=('actual_amount_paid','sum'),
      avg_list_price=('plan_list_price','mean'),
      avg_discount_rate=('discount_rate','mean'),
      last_txn_date=('transaction_date','max'),
      last_expire_date=('membership_expire_date','max')
    )
).reset_index()

In [43]:
txn_agg['last_gap_days'] = (txn_agg['last_expire_date'] - txn_agg['last_txn_date']).dt.days

- ngày hết hạn của lần đó - ngày thanh toán cuối = .. gap

### **Nhóm 1: Các tín hiệu "Báo động đỏ" (Churn Signals)**
*Đây là những biến có tương quan trực tiếp và mạnh mẽ nhất đến việc rời bỏ ứng dụng.*

+ cancel_rate (Tỷ lệ bấm hủy): Khách hàng nào có lịch sử thường xuyên bấm hủy gói (rate > 0) là những đối tượng thiếu ổn định, nguy cơ Churn rất lớn.

+ last_gap_days (Khoảng cách hết hạn): Biến để bắt các trường hợp "Hủy ngang". Bình thường khoảng cách này dương (hết hạn sau ngày mua). Nếu nó âm, cho thấy việc tài khoản đã bị hệ thống cắt ép buộc trước hạn.

### **Nhóm 2: Chỉ số gắn kết & Trung thành (Loyalty & Stickiness)**
*Nhóm biến này giúp mô hình nhận diện tập khách hàng an toàn.*

+ auto_renew_rate (Tỷ lệ bật tự động gia hạn): Biến quan trọng bậc nhất. Khách hàng có rate tiến gần đến 1.0 (luôn bật gia hạn) là những khách tiềm năng, hiếm khi Churn trừ khi thẻ tín dụng của họ bị lỗi.

+ n_txns (Tổng số giao dịch): Thể hiện thâm niên. Người nạp tiền 20 lần chắc chắn gắn bó hơn người mới nạp 1 lần.

+ share_30d (Tỷ lệ dùng gói chuẩn 30 ngày): Phân biệt giữa người dùng ổn định (mua gói tháng đều đặn) và người dùng qua - kiểu dùng tạm 1 lúc (mua gói 7 ngày, 14 ngày).

### **Nhóm 3: Độ nhạy cảm với giá trị (Price Sensitivity)**
*Phân tích tệp khách hàng săn khuyến mãi.*

+ avg_discount_rate (Tỷ lệ giảm giá trung bình): Nhận diện nhóm "khách hàng săn Sale". Nếu tỷ lệ này cao, chứng tỏ khách hàng bị thu hút bởi giá rẻ. Khi hết khuyến mãi, nhóm này có xác suất Churn rất cao.

+ zero_paid_rate (Tỷ lệ dùng miễn phí): Bắt được nhóm khách hàng xài app qua các chương trình tặng kèm hoặc khuyến mãi 100%. Nhóm này ít khi chuyển đổi thành khách hàng trả phí thực sự.

### **Nhóm 4: Sự ổn định trong hành vi (Behavioral Variance)**
+ std_plan_days (Độ lệch chuẩn số ngày mua): Đo lường sự bất thường. Nếu bằng 0, khách hàng luôn mua 1 loại gói (rất ổn định). Nếu chỉ số này cao, khách hàng thường xuyên nhảy qua lại giữa gói ngắn hạn và dài hạn, cho thấy họ đang do dự về việc gắn bó lâu dài.

PHẦN B — Aggregate user_logs_v2

In [44]:
logs = user_v2.copy()

- Aggregate

In [45]:
user_v2['date'].max()

Timestamp('2017-03-31 00:00:00')

In [46]:
# ==========================================
# 1. TIỀN XỬ LÝ (Tạo biến phụ trên từng dòng)
# ==========================================
logs['total_listens'] = logs['num_25'] + logs['num_50'] + logs['num_75'] + logs['num_985'] + logs['num_100']
logs['complete_listens'] = logs['num_985'] + logs['num_100']


# ==========================================
# 2. GOM NHÓM (Aggregation)
# ==========================================
logs_agg = (logs.groupby('msno').agg(
    active_days=('date', 'nunique'),
    last_activity_date=('date', 'max'),
    total_secs_sum=('total_secs', 'sum'),
    max_unique_songs_per_day=('num_unq', 'max'),
    
    # Các biến thô dùng để tính tỷ lệ ở bước sau
    total_listens=('total_listens', 'sum'),
    complete_listens=('complete_listens', 'sum'),
    sum_num_unq=('num_unq', 'sum'),
    
    # [MỚI] Tổng số lần skip sớm dưới 25% thời lượng
    sum_num_25=('num_25', 'sum')
)).reset_index()


# ==========================================
# 3. TẠO ĐẶC TRƯNG TỶ LỆ (Feature Engineering)
# ==========================================
# Trung bình giây nghe mỗi ngày
logs_agg['total_secs_mean'] = np.where(
    logs_agg['active_days'] > 0,
    logs_agg['total_secs_sum'] / logs_agg['active_days'],
    0
)

# Tỷ lệ nghe trọn vẹn bài hát (Chất lượng trải nghiệm)
logs_agg['completion_ratio'] = np.where(
    logs_agg['total_listens'] > 0,
    logs_agg['complete_listens'] / logs_agg['total_listens'],
    0
)

# [MỚI] Tỷ lệ Bực bội / Chán nản (Skip ngay khi vừa bật)
logs_agg['early_skip_rate'] = np.where(
    logs_agg['total_listens'] > 0,
    logs_agg['sum_num_25'] / logs_agg['total_listens'],
    0
)

# [MỚI] Tỷ lệ Hoài niệm (Thích nghe lại nhạc cũ quen thuộc)
logs_agg['repeat_ratio'] = np.where(
    logs_agg['total_listens'] > 0,
    1 - (logs_agg['sum_num_unq'] / logs_agg['total_listens']),
    0
)


# ==========================================
# 4. DỌN RÁC (Chống Overfitting & Đa cộng tuyến)
# ==========================================
# Xóa bỏ các cục số liệu tuyệt đối thô ráp, chỉ giữ lại Tỷ lệ (Rates & Ratios)
cols_to_drop = [
    'total_secs_sum',    # Đã có total_secs_mean
    'total_listens',     # Đã quy ra các loại Ratio
    'complete_listens',  # Đã có completion_ratio
    'sum_num_unq',       # Đã có repeat_ratio
    'sum_num_25'         # Đã có early_skip_rate
]
logs_agg = logs_agg.drop(columns=cols_to_drop)

print("Kích thước bảng Logs sau xử lý:", logs_agg.shape)
display(logs_agg.head())

Kích thước bảng Logs sau xử lý: (316345, 8)


,msno,active_days,last_activity_date,max_unique_songs_per_day,total_secs_mean,completion_ratio,early_skip_rate,repeat_ratio
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,1,2017-03-25,23,6182.491000,0.888889,0.111111,0.148148
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,1,2017-03-01,19,4140.721000,0.800000,0.200000,0.050000
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2,2017-03-09,32,7695.039000,0.904110,0.054795,0.273973
3,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,3,2017-03-18,12,2487.311667,0.761905,0.166667,0.571429
4,++0+IdHga8fCSioOVpU8K7y4Asw8AveIApVH2r9q9yY=,4,2017-03-27,72,4781.065000,0.541985,0.343511,0.030534


In [47]:
# logs_agg[['total_secs_mean','avg_unique_songs_per_day','completion_ratio']] = logs_agg[
# ['total_secs_mean','avg_unique_songs_per_day','completion_ratio']].fillna(0)

**1: Độ nghiện**
+ active_days (Số ngày hoạt động): Biến số Quan trọng nhất (Top 1 Feature) của cả dự án. Trong sản phẩm âm nhạc, không gì phản ánh sự trung thành tốt bằng việc khách hàng mở app bao nhiêu ngày trong 1 tháng. Dưới 10 ngày $\rightarrow$ Nguy cơ Churn cực cao.
+ total_secs_mean (Thời gian nghe trung bình/ngày): Đo lường "độ sâu" của việc sử dụng. Có những người mở app mỗi ngày nhưng chỉ nghe 1 bài rồi tắt, khác hoàn toàn với người cắm tai nghe 5 tiếng/ngày.

**2: Chất lượng Trải nghiệm (Quality of Experience)**
+ completion_ratio (Tỷ lệ nghe trọn vẹn): Vũ khí bí mật. Nếu chỉ số này tiến về 0, nghĩa là khách hàng liên tục bấm "Next" (Bỏ qua). Đây là biểu hiện của sự bực bội, thuật toán gợi ý nhạc dở tệ, hoặc không có nhạc đúng gu. Churn là điều chắc chắn.** 3: Thói quen & Sự khám phá (Habit & Discovery)**
+ avg_unique_songs_per_day (Số bài hát mới trung bình): Phân loại người dùng thành 2 nhóm:
    + Nhóm hoài cổ (Điểm thấp): Chỉ bật đi bật lại 1 playlist quen thuộc.Nhóm khám phá (Điểm cao): Liên tục tìm kiếm bài hát mới.
    + Nhóm này dễ tính tiền, nhưng cũng rất dễ rời đi nếu kho nhạc của KKBox không cập nhật nhanh bằng Spotify.

In [48]:
train_v2.info()

<class 'pandas.DataFrame'>
RangeIndex: 970960 entries, 0 to 970959
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   msno      970960 non-null  str  
 1   is_churn  970960 non-null  int64
dtypes: int64(1), str(1)
memory usage: 55.6 MB


In [49]:
txn_agg.info()

<class 'pandas.DataFrame'>
RangeIndex: 1197050 entries, 0 to 1197049
Data columns (total 14 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   msno               1197050 non-null  str           
 1   n_txns             1197050 non-null  int64         
 2   cancel_rate        1197050 non-null  float64       
 3   auto_renew_rate    1197050 non-null  float64       
 4   avg_plan_days      1197050 non-null  float64       
 5   share_30d          1197050 non-null  float64       
 6   zero_paid_rate     1197050 non-null  float64       
 7   avg_amount_paid    1197050 non-null  float64       
 8   total_amount_paid  1197050 non-null  int64         
 9   avg_list_price     1197050 non-null  float64       
 10  avg_discount_rate  1195695 non-null  float64       
 11  last_txn_date      1197050 non-null  datetime64[us]
 12  last_expire_date   1197050 non-null  datetime64[us]
 13  last_gap_days      1197050 non-null  i

In [50]:
logs_agg.info()

<class 'pandas.DataFrame'>
RangeIndex: 316345 entries, 0 to 316344
Data columns (total 8 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   msno                      316345 non-null  str           
 1   active_days               316345 non-null  int64         
 2   last_activity_date        316345 non-null  datetime64[us]
 3   max_unique_songs_per_day  316345 non-null  int64         
 4   total_secs_mean           316345 non-null  float64       
 5   completion_ratio          316345 non-null  float64       
 6   early_skip_rate           316345 non-null  float64       
 7   repeat_ratio              316345 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(2), str(1)
memory usage: 32.6 MB


In [51]:
members_v3.info()

<class 'pandas.DataFrame'>
RangeIndex: 6769473 entries, 0 to 6769472
Data columns (total 6 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   msno                    str           
 1   city                    category      
 2   bd                      float64       
 3   gender                  category      
 4   registered_via          category      
 5   registration_init_time  datetime64[us]
dtypes: category(3), datetime64[us](1), float64(1), str(1)
memory usage: 458.4 MB


## **Merge data**

In [52]:
final_df = (train_v2.merge( members_v3,on='msno',how='left')
.merge(
   txn_agg,
   on='msno',
   how='left'
 )
 .merge(
   logs_agg,
   on='msno',
   how='left'
 )
)

In [53]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 970960 entries, 0 to 970959
Data columns (total 27 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   msno                      970960 non-null  str           
 1   is_churn                  970960 non-null  int64         
 2   city                      860967 non-null  category      
 3   bd                        386612 non-null  float64       
 4   gender                    860967 non-null  category      
 5   registered_via            860967 non-null  category      
 6   registration_init_time    860967 non-null  datetime64[us]
 7   n_txns                    933578 non-null  float64       
 8   cancel_rate               933578 non-null  float64       
 9   auto_renew_rate           933578 non-null  float64       
 10  avg_plan_days             933578 non-null  float64       
 11  share_30d                 933578 non-null  float64       
 12  zero_paid_rat

In [54]:
final_df.head()

,msno,is_churn,city,bd,gender,registered_via,registration_init_time,n_txns,cancel_rate,auto_renew_rate,...,last_txn_date,last_expire_date,last_gap_days,active_days,last_activity_date,max_unique_songs_per_day,total_secs_mean,completion_ratio,early_skip_rate,repeat_ratio
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,5,28.0,male,3,2013-12-23,NaN,NaN,NaN,...,NaT,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,13,20.0,male,3,2013-12-23,1.0,0.000,0.0,...,2017-03-11,2017-03-31,20.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,13,18.0,male,3,2013-12-27,2.0,0.000,0.0,...,2017-03-14,2017-03-31,17.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,1,NaN,Unknown,7,2014-01-09,10.0,0.000,1.0,...,2015-12-08,2017-03-31,479.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1,13,35.0,female,7,2014-01-25,8.0,0.125,1.0,...,2017-03-16,2017-03-31,15.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN


In [55]:
print(final_df["registration_init_time"].max())
print(final_df["last_txn_date"].max())
print(final_df["last_expire_date"].max())
print(final_df["last_activity_date"].max())

2017-04-24 00:00:00
2017-03-31 00:00:00
2017-03-31 00:00:00
2017-03-31 00:00:00


In [56]:
# Kiểm tra tỷ lệ phần trăm thiếu
missing_data = final_df.isnull().sum()
missing_percentage = (missing_data / len(final_df)) * 100
missing_table = pd.DataFrame({'Missing Count': missing_data, 'Percentage': missing_percentage})
missing_table = missing_table[missing_table['Percentage'] > 0].sort_values(by='Percentage', ascending=False)

print("Các cột có giá trị thiếu:")
print(missing_table)

Các cột có giá trị thiếu:
                          Missing Count  Percentage
repeat_ratio                     738799   76.089540
early_skip_rate                  738799   76.089540
completion_ratio                 738799   76.089540
total_secs_mean                  738799   76.089540
max_unique_songs_per_day         738799   76.089540
last_activity_date               738799   76.089540
active_days                      738799   76.089540
bd                               584348   60.182500
city                             109993   11.328273
registration_init_time           109993   11.328273
registered_via                   109993   11.328273
gender                           109993   11.328273
avg_discount_rate                 38071    3.920965
share_30d                         37382    3.850004
zero_paid_rate                    37382    3.850004
avg_amount_paid                   37382    3.850004
avg_plan_days                     37382    3.850004
avg_list_price                    3738

In [57]:
(final_df['avg_discount_rate'] > 1).sum()

np.int64(0)

In [58]:
(final_df['last_gap_days']<0).sum()

np.int64(2178)

In [59]:
(final_df['completion_ratio']>1).sum()

np.int64(0)

In [60]:
final_df['total_secs_mean'].describe(
 percentiles=[.95,.99]
)

count    232161.000000
mean       7604.856173
std        9142.662910
min           0.078000
95%       26480.334000
99%       42247.941200
max      289164.798000
Name: total_secs_mean, dtype: float64

In [61]:
numeric_cols = final_df.select_dtypes(include = ['number']).columns
corr_matrix = final_df[numeric_cols].corr()
threshold = 0.9
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column].abs() > threshold)]

print(f"Các cột có tương quan > {threshold} (nên cân nhắc xóa): {to_drop}")

Các cột có tương quan > 0.9 (nên cân nhắc xóa): ['avg_amount_paid', 'avg_list_price']


In [62]:
# 1. Danh sách các cột gây Đa cộng tuyến (Trùng lặp thông tin)
redundant_cols = [
    'avg_list_price',     # Đã có total_amount_paid và avg_discount_rate
    'avg_amount_paid',    # Tương tự như trên
]

# 2. Danh sách các cột Nhân khẩu học gây nhiễu (Ít giá trị dự đoán Churn)
# Nếu sau này mô hình thiếu chính xác, thử bỏ comment để thêm lại chúng
# low_importance_cols = [
    # 'gender',             # 65% là Unknown, đưa vào chỉ làm chậm mô hình
    # 'city',               # Ít tác động bằng hành vi nghe nhạc thực tế
    # 'registered_via'      # Ít tác động
# ]

# Gộp danh sách và Drop khỏi DataFrame
cols_to_drop = redundant_cols #+ low_importance_cols
df_final = final_df.drop(columns=cols_to_drop)

print(f"Đã drop {len(cols_to_drop)} cột rác.")

Đã drop 2 cột rác.


In [63]:
cols_to_fill = ['gender', 'city', 'registered_via']

for col in cols_to_fill:
    # 1. Chuyển sang dạng object để không bị chặn bởi constraints của category
    df_final[col] = df_final[col].astype(object)
    
    # 2. Xử lý NaN và các giá trị lỗi thành 'Unknown'
    df_final[col] = df_final[col].fillna('Unknown')
    df_final[col] = df_final[col].astype(str).replace(['nan', '-1', -1], 'Unknown')
    
    # 3. Sau khi đã làm sạch xong, mới chuyển lại thành category
    df_final[col] = df_final[col].astype('category')

# Kiểm tra lại
print(df_final[cols_to_fill].isnull().sum())

gender            0
city              0
registered_via    0
dtype: int64


In [64]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 970960 entries, 0 to 970959
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   msno                      970960 non-null  str           
 1   is_churn                  970960 non-null  int64         
 2   city                      970960 non-null  category      
 3   bd                        386612 non-null  float64       
 4   gender                    970960 non-null  category      
 5   registered_via            970960 non-null  category      
 6   registration_init_time    860967 non-null  datetime64[us]
 7   n_txns                    933578 non-null  float64       
 8   cancel_rate               933578 non-null  float64       
 9   auto_renew_rate           933578 non-null  float64       
 10  avg_plan_days             933578 non-null  float64       
 11  share_30d                 933578 non-null  float64       
 12  zero_paid_rat

In [65]:
# Mốc thời gian tham chiếu cho dự án KKBox
ref_date = pd.to_datetime('2017-04-01')

# 1. Trích xuất thông tin thâm niên và độ mới
df_final['days_since_reg'] = (ref_date - df_final['registration_init_time']).dt.days
df_final['days_since_last_activity'] = (ref_date - df_final['last_activity_date']).dt.days
df_final['days_since_last_txn'] = (ref_date - df_final['last_txn_date']).dt.days

# 2. Xử lý các giá trị NaN phát sinh (Ví dụ khách chưa bao giờ nghe nhạc)
df_final['days_since_last_activity'] = df_final['days_since_last_activity'].fillna(999)

# 3. GIỜ MỚI XÓA (Bắt buộc)
cols_to_drop = ['registration_init_time', 'last_activity_date', 'last_txn_date', 'last_expire_date']
df_final = df_final.drop(columns=cols_to_drop)

print("Đã chuyển đổi Datetime thành Recency và xóa các cột gốc.")

Đã chuyển đổi Datetime thành Recency và xóa các cột gốc.


- registration_init_time: Chuyển thành days_since_registration. Khách hàng mới (Newbie) thường có hành vi rời bỏ rất khác so với khách hàng lâu năm (Loyalists).

- last_activity_date: Chuyển thành days_since_last_activity. Đây là chỉ số "nóng" nhất. Khách hàng không nghe nhạc trong 15-20 ngày gần nhất có xác suất rời bỏ cực cao.
    +  Nếu dùng fillna(0): Số 0 mang ý nghĩa là "Khoảng cách bằng 0", tức là khách hàng vừa mới mở app ngày hôm nay. Mô hình sẽ bị lú lấp tức: Tại sao một người mới vào app hôm nay lại có xác suất Churn cao thế nhỉ? $\rightarrow$ Dự đoán sai !

    +  Nếu dùng fillna(mean) hoặc fillna(median) (VD: 14 ngày): Bạn đang cào bằng những "khách hàng ma" này chung mâm với những khách hàng bình thường (khoảng 2 tuần mở app 1 lần). Mô hình sẽ không thể phân biệt được ai là người đang dùng bình thường và ai là người đã bỏ app.
    
    +  Khi dùng fillna(999): Bạn đang đẩy nhóm khách hàng lâu này ra một nơi xa riêng biệt ở tận cùng phía bên phải của trục số.

- last_txn_date: Chuyển thành days_since_last_txn. Đo lường mức độ gần đây của việc giao dịch tài chính.

- last_expire_date: Dùng để so sánh với ngày thanh toán cuối cùng (last_gap_days đã tạo).

In [66]:
df_final.isnull().sum()

msno                             0
is_churn                         0
city                             0
bd                          584348
gender                           0
registered_via                   0
n_txns                       37382
cancel_rate                  37382
auto_renew_rate              37382
avg_plan_days                37382
share_30d                    37382
zero_paid_rate               37382
total_amount_paid            37382
avg_discount_rate            38071
last_gap_days                37382
active_days                 738799
max_unique_songs_per_day    738799
total_secs_mean             738799
completion_ratio            738799
early_skip_rate             738799
repeat_ratio                738799
days_since_reg              109993
days_since_last_activity         0
days_since_last_txn          37382
dtype: int64

In [67]:
from sklearn.model_selection import train_test_split

# Giữ nguyên logic chia dữ liệu của bạn
X = df_final.drop(columns=['msno', 'is_churn'])
y = df_final['is_churn']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ĐẢM BẢO KHÔNG LEAKAGE: 
# Tính giá trị để lấp đầy (median) CHỈ trên X_train
median_bd = X_train['bd'].median()
median_reg = X_train['days_since_reg'].median()

# Lấp đầy cho cả hai tập dựa trên median của tập train
X_train['bd'] = X_train['bd'].fillna(median_bd)
X_val['bd'] = X_val['bd'].fillna(median_bd)

X_train['days_since_reg'] = X_train['days_since_reg'].fillna(median_reg)
X_val['days_since_reg'] = X_val['days_since_reg'].fillna(median_reg)

In [68]:
from sklearn.model_selection import train_test_split

print("Kích thước dữ liệu ban đầu:", df_final.shape)

# 1 & 2: TRUE ZEROS & MATH UNDEFINED
zero_fill_cols = [
    # Từ logs 
    'active_days', 'max_unique_songs_per_day', 'total_secs_mean', 'completion_ratio', 
    'early_skip_rate', 'repeat_ratio',
    # Từ transactions
    'n_txns', 'cancel_rate', 'auto_renew_rate', 'avg_plan_days', 'share_30d', 
    'zero_paid_rate', 'total_amount_paid', 'avg_discount_rate', 'last_gap_days',
    'has_payment_friction' 
]

existing_zero_cols = [col for col in zero_fill_cols if col in df_final.columns]
df_final[existing_zero_cols] = df_final[existing_zero_cols].fillna(0)

if 'days_since_last_txn' in df_final.columns:
    df_final['days_since_last_txn'] = df_final['days_since_last_txn'].fillna(999)


#  SPLIT TRAIN / VAL
X = df_final.drop(columns=['msno', 'is_churn'])
y = df_final['is_churn']

print("Đang chia dữ liệu Train/Val (Tỷ lệ 80/20)...")
# ĐỔI TÊN BIẾN THÀNH X_val, y_val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


#  4: ĐIỀN KHUYẾT THỐNG KÊ (IMPUTATION)
# Tính Trung vị CHỈ TRÊN TẬP TRAIN
median_bd = X_train['bd'].median()
median_reg = X_train['days_since_reg'].median()

# Lấp chỗ trống cho cả Train và Val (Chống Leakage)
X_train['bd'] = X_train['bd'].fillna(median_bd)
X_val['bd'] = X_val['bd'].fillna(median_bd)

X_train['days_since_reg'] = X_train['days_since_reg'].fillna(median_reg)
X_val['days_since_reg'] = X_val['days_since_reg'].fillna(median_reg)

print("\n--- HOÀN TẤT CHUẨN BỊ DỮ LIỆU ---")
print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_val: {X_val.shape}")
print("Số NaN còn lại trong X_train:", X_train.isna().sum().sum())
print("Số NaN còn lại trong X_val:", X_val.isna().sum().sum())

Kích thước dữ liệu ban đầu: (970960, 24)
Đang chia dữ liệu Train/Val (Tỷ lệ 80/20)...

--- HOÀN TẤT CHUẨN BỊ DỮ LIỆU ---
Kích thước X_train: (776768, 22)
Kích thước X_val: (194192, 22)
Số NaN còn lại trong X_train: 0
Số NaN còn lại trong X_val: 0


In [69]:
X_train.to_parquet("Data/X_train.parquet")
X_val.to_parquet("Data/X_val.parquet")
y_train.to_frame(name='is_churn').to_parquet("Data/y_train.parquet")
y_val.to_frame(name='is_churn').to_parquet("Data/y_val.parquet")